> **Note:** This notebook requires run outputs that are not included in the published repository. To reproduce, re-run the pipeline with the appropriate configuration to generate the required analysis DataFrames.
>
> This notebook references FINAL_COMMS_BASELINES* runs which must be regenerated via the pipeline.


In [1]:
import pandas as pd 
import numpy as np
import constants as c
from glob import glob
from scipy import stats 

In [2]:
# Glob pattern for all baseline performance files
CURRENT_PP_BASELINES_GLOB = "../../runs/icar_icar/simulated_False/ahl_True/covariates_True/FINAL_COMMS_BASELINES*/performance_on_baselines.csv"

# Find all matching files
baseline_files = sorted(glob(CURRENT_PP_BASELINES_GLOB))
print(f"Found {len(baseline_files)} baseline performance files:")

Found 20 baseline performance files:


In [3]:
# Read all baseline performance files
all_baselines = []
for f in baseline_files:
    df = pd.read_csv(f)
    df = df.set_index(df.columns[0])
    all_baselines.append(df)
    
print(f"Loaded {len(all_baselines)} dataframes")

Loaded 20 dataframes


In [4]:
# Stack all dataframes into a 3D array for aggregation
stacked = np.stack([df.values for df in all_baselines], axis=0)

# Compute mean and standard error
mean_values = np.mean(stacked, axis=0)
std_values = np.std(stacked, axis=0, ddof=1)
n_samples = len(all_baselines)

# 95% confidence interval (t-distribution)
t_crit = stats.t.ppf(0.975, df=n_samples - 1)
ci_half_width = t_crit * (std_values / np.sqrt(n_samples))

# Create dataframes for mean and CI
pp_baselines_mean = pd.DataFrame(mean_values, index=all_baselines[0].index, columns=all_baselines[0].columns)
pp_baselines_std = pd.DataFrame(std_values, index=all_baselines[0].index, columns=all_baselines[0].columns)
pp_baselines_ci = pd.DataFrame(ci_half_width, index=all_baselines[0].index, columns=all_baselines[0].columns)

print(f"Aggregated {n_samples} runs")
print(f"95% CI t-critical value: {t_crit:.3f}")

Aggregated 20 runs
95% CI t-critical value: 2.093


In [5]:
# Create formatted dataframe with mean ± CI
def format_with_ci(mean_df, ci_df, decimals=2):
    """Format values as 'mean ± ci' strings."""
    formatted = pd.DataFrame(index=mean_df.index, columns=mean_df.columns)
    for col in mean_df.columns:
        for idx in mean_df.index:
            mean_val = mean_df.loc[idx, col]
            ci_val = ci_df.loc[idx, col]
            formatted.loc[idx, col] = f"{mean_val:.{decimals}f} ± {ci_val:.{decimals}f}"
    return formatted

pp_baselines_formatted = format_with_ci(pp_baselines_mean, pp_baselines_ci)

# Also keep a numeric version for highlighting (use mean values)
pp_baselines = pp_baselines_mean.round(2)

groups = { 
    'frac_positive_classifications': ['frac_positive_classifications'],
    'any_positive_classifications': ['any_positive_classifications'],
    'n_positive_classifications': ['n_positive_classifications'],
    'any_positive_ground_truth': ['any_positive_ground_truth'],
    'n_positive_ground_truth': ['n_positive_ground_truth'],
    'graph_laplacian':  # grab any rows that contain 'graph_laplacian'
        [row for row in pp_baselines.index if 'graph_laplacian' in row],
    'OLS': # grab any rows that contain 'OLS'
        [row for row in pp_baselines.index if 'OLS' in row],
    'RandomForest': # grab any rows that contain 'RandomForest'
        [row for row in pp_baselines.index if 'RandomForest' in row],
    'bayesian_model': # grab any rows that contain 'bayesian_model'
        [row for row in pp_baselines.index if 'bayesian_model' in row],
}



In [6]:
# create a styling function that highlights based on mean values
def highlight_max_by_group(df):
    # create empty style DataFrame
    styles = pd.DataFrame('', index=df.index, columns=df.columns)
    
    # for each group of rows
    for group_name, rows in groups.items():
        if not rows:
            continue
            
        # for each column
        for col in df.columns:
            # get max value for this group in this column (using mean values)
            max_val = pp_baselines_mean.loc[rows, col].max()
            # set style for matching values in this group
            for row in rows:
                if pp_baselines_mean.loc[row, col] == max_val:
                    styles.loc[row, col] = 'background-color: darkgreen'
    
    return styles

# apply the styling to the formatted dataframe (with ± notation)
styled_baselines = pp_baselines_formatted.style.apply(highlight_max_by_group, axis=None)

# Also create a numeric-only styled version
styled_baselines_numeric = pp_baselines.style.apply(highlight_max_by_group, axis=None)
styled_baselines_numeric = styled_baselines_numeric.format("{:.2f}")

In [7]:
print("Results with 95% Confidence Intervals (mean ± CI):")
styled_baselines

Results with 95% Confidence Intervals (mean ± CI):


,"pearson r, frac_positive_classifications","AUC, any ground truth positive","AUC, any classified positive"
Unnamed: 0,,,
frac_positive_classifications,0.56 ± 0.03,0.75 ± 0.01,0.67 ± 0.00
any_positive_classifications,0.26 ± 0.01,0.75 ± 0.01,0.67 ± 0.00
n_positive_classifications,0.30 ± 0.01,0.76 ± 0.01,0.68 ± 0.00
any_positive_ground_truth,0.27 ± 0.02,0.62 ± 0.01,0.57 ± 0.00
n_positive_ground_truth,0.27 ± 0.02,0.62 ± 0.01,0.57 ± 0.00
graph_laplacian_frac_pos_classifications_one_iter,0.56 ± 0.03,0.78 ± 0.01,0.69 ± 0.01
graph_laplacian_frac_pos_classifications_five_iter,0.57 ± 0.03,0.77 ± 0.01,0.69 ± 0.00
graph_laplacian_n_positive_ground_truth_one_iter,0.27 ± 0.02,0.64 ± 0.01,0.59 ± 0.00
graph_laplacian_n_positive_ground_truth_five_iter,0.27 ± 0.02,0.66 ± 0.01,0.60 ± 0.01


In [8]:
# Display numeric-only version (mean values)
print("Mean values only:")
styled_baselines_numeric

Mean values only:


,"pearson r, frac_positive_classifications","AUC, any ground truth positive","AUC, any classified positive"
Unnamed: 0,,,
frac_positive_classifications,0.56,0.75,0.67
any_positive_classifications,0.26,0.75,0.67
n_positive_classifications,0.30,0.76,0.68
any_positive_ground_truth,0.27,0.62,0.57
n_positive_ground_truth,0.27,0.62,0.57
graph_laplacian_frac_pos_classifications_one_iter,0.56,0.78,0.69
graph_laplacian_frac_pos_classifications_five_iter,0.57,0.77,0.69
graph_laplacian_n_positive_ground_truth_one_iter,0.27,0.64,0.59
graph_laplacian_n_positive_ground_truth_five_iter,0.27,0.66,0.60
